In [1]:
# Memuat API key dari file .env
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Mendefinisikan prompt secara manual menggunakan format role-based tuple
system_prompt = """
Kamu adalah customer service asistant yang menghadapi keluhan permasalahan
pelanggan.
"""
customer_email = """
Permisi Mas. Saya ingin menyampaikan keluhan tentang makanan yang sudah saya pesan. Saya tadi memesan nasi goreng, namun tidak diberikan telur mata sapi seperti rincian pesanan saya. Saya juga memesan minuman namun belum datang.
"""

prompt = """Berdasarkan kalimat ini, apa sentimen yang disampaikan? Apakah positif, netral, atau negatif?"""
message = prompt + customer_email

# Inisialisasi model Google GenAI dengan temperature 0 agar jawaban lebih konsisten/faktual
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)
# Menyusun pesan sebagai tuple berisi (role, content)
messages = [( "system", system_prompt),( "human", message)]
ai_msg = llm.invoke(messages)
print( ai_msg.content)

[{'type': 'text', 'text': 'Sentimen dari kalimat tersebut adalah **negatif**.\n\n**Penjelasan:**\nPelanggan menyampaikan ketidakpuasan karena adanya kesalahan dalam pesanan (telur mata sapi yang tidak ada) dan keterlambatan layanan (minuman yang belum datang). Meskipun disampaikan dengan bahasa yang sopan ("Permisi Mas"), isi pesannya tetap merupakan sebuah keluhan atas layanan yang tidak sesuai.\n\n---\n\nSebagai asisten *customer service*, berikut adalah contoh respons yang tepat untuk menangani keluhan tersebut:\n\n*"Mohon maaf sekali atas ketidaknyamanannya, Kak. Kami sangat menyesal atas kesalahan pada pesanan nasi gorengnya dan keterlambatan minuman yang Kakak pesan. Boleh saya tahu nomor pesanannya agar segera saya bantu cek dan tindak lanjuti ke bagian dapur/bar kami? Kami akan segera mengirimkan kekurangan pesanannya sekarang juga."*', 'extras': {'signature': 'EjQKMgERTTIPBk7euzDVGdih5U2FqHAQjIrpVH3Hra2nJOHe2WdwGXIPasrc9KCoMw92ScZK'}}]


## Structured Prompt

In [4]:
# Menggunakan ChatPromptTemplate untuk prompt terstruktur berbasis chat
from langchain_core.prompts import ChatPromptTemplate

# 1. Definisikan keluhan pelanggan
customer_message = """Permisi Mas. Saya ingin menyampaikan keluhan tentang makanan yang sudah saya pesan. Saya tadi memesan nasi goreng, namun tidak diberikan telur mata sapi seperti rincian pesanan saya. Saya juga memesan minuman namun belum datang."""

# 2. Buat ChatPromptTemplate dengan pembagian peran (system & human)
chat_template = ChatPromptTemplate.from_messages([
    ("system", "Kamu adalah asisten analisis sentimen restoran yang bertugas mengklasifikasikan pesan pelanggan."),
    ("human", """Dibawah ini adalah keluhan dari pelanggan restoran:
    
    {content}
    
    Berdasarkan kalimat ini, apa sentimen yang disampaikan? Apakah positif, netral, atau negatif?""")
])

# 3. Format variabel ke dalam template pesan
formatted_messages = chat_template.format_messages(content=customer_message)

# 4. Kirim daftar pesan terstruktur ke model
response = llm.invoke(formatted_messages)

print(response.content)

[{'type': 'text', 'text': 'Sentimen dari pesan pelanggan tersebut adalah **negatif**.\n\n**Analisis:**\nPelanggan menyampaikan keluhan terkait dua masalah spesifik:\n1. **Kesalahan pesanan:** Nasi goreng yang diterima tidak sesuai dengan rincian (kurang telur mata sapi).\n2. **Pelayanan yang lambat/terlupa:** Minuman yang dipesan belum diantarkan.\n\nMeskipun bahasa yang digunakan pelanggan sopan ("Permisi Mas"), isi pesannya secara keseluruhan menunjukkan ketidakpuasan terhadap kualitas layanan dan akurasi pesanan restoran.', 'extras': {'signature': 'EjQKMgERTTIPas9O05BJWoZ5TAZdYBD97u/J3gZJ01wn+Dn821Wx6u7K9fdy2QM3ojoVTNYn'}}]


## Structured Output

Secara default, LLM menghasilkan output berupa teks bebas. Namun, seringkali kita membutuhkan output yang terstruktur (seperti JSON atau objek data) agar bisa diproses lebih lanjut oleh program. 

In [11]:
from pydantic import BaseModel
from langchain.agents import create_agent
class HasilAnalisis(BaseModel):
    sentimen: str  # "positif" | "netral" | "negatif"
    kategori: str  # contoh: "makanan", "pengiriman", "pelayanan"
    urgensi: str   # "rendah" | "sedang" | "tinggi"
    
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    response_format=HasilAnalisis,
    system_prompt="Analisis keluhan pelanggan restoran.",
)

question = "Permisi Mas. Saya ingin menyampaikan keluhan tentang makanan yang sudah saya pesan. Saya tadi memesan nasi goreng, namun tidak diberikan telur mata sapi seperti rincian pesanan saya. Saya juga memesan minuman namun belum datang."
response = agent.invoke({"messages": [question]})
print(response["structured_response"])

# HasilAnalisis(sentimen='negatif', kategori='makanan', urgensi='sedang')

sentimen='negatif' kategori='kesalahan pesanan dan layanan' urgensi='tinggi'
